**20 | HARTH Research Evidence Consolidation and Provenance Audit**

**Human Movement Intelligence | Research portfolio | October 2026**

This notebook consolidates the complete 19-stage HARTH v2.0 development history without retraining any model. It cross-references original executed notebooks with a transparent evidence register, audits saved execution state, separates matched from non-matched comparisons, documents uncertainty and test-set limitations, and prepares a defensible research narrative for the remaining stages.

**Scope and scientific boundary:** 29 development subjects support later experiments. `S023` and `S032` were designated as reserved model-development Test subjects after earlier exploratory inspection. They are **not** a strictly preregistered unseen Test set. Numerical findings quoted in the evidence register are transcribed from earlier saved outputs and must be checked against those sources, not treated as recomputed statistics.

**01. Study objectives and evidence hierarchy**

**Questions answered:** What has been implemented? Which experimental comparisons share the same participants, window sampling and evaluation protocol? Which conclusions are still unsupported? What must be frozen before final evaluation?

**Evidence hierarchy:** (1) Executed notebooks with saved results and their source code; (2) reusable Python functions and explicit tests; (3) protocol Markdown files; (4) the manually curated experiment register. A GitHub PASS marker confirms a stored quality-gate message, not generalisation. This audit does not download the full HARTH dataset or reconstruct scores.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Locate the repository and load the shared read-only evidence functions.
root = Path.cwd().resolve()
while not (root / "code/python/src/core/notebook_evidence.py").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside human-movement-intelligence.")
    root = root.parent
sys.path.insert(0, str(root / "code/python/src"))
from core.notebook_evidence import (
    collect_notebook_inventory, validate_evidence_register,
    extract_saved_output_text,
)
notebook_dir = root / "studies/wearable_activity_recognition/notebooks"
register_path = root / "studies/wearable_activity_recognition/experiments/stage_01_19_evidence_register.csv"
print("Repository:", root)
print("Source notebooks:", notebook_dir)
print("Evidence register:", register_path)

**02. Inspect the saved execution state of all 19 notebooks**

A reproducibility review must distinguish between notebook source code, saved output, and a newly executed run. The inventory includes code cell counts, missing execution counts, saved error objects, output-bearing cells, and whether a saved output contains `PASSED`. Missing counts or errors are shown explicitly; no missing outputs are manufactured.

In [ ]:
# Inspect notebook JSON without executing expensive HARTH training pipelines.
inventory = collect_notebook_inventory(notebook_dir)
display(inventory[[
    "stage", "notebook", "code_cells", "executed_code_cells",
    "cells_with_outputs", "unexecuted_code_cells",
    "saved_error_count", "saved_pass_marker",
]])
print("Stages inventoried:", len(inventory))
print("Saved errors:", int(inventory["saved_error_count"].sum()))
print("Stages with incomplete execution counts:",
      inventory.loc[inventory["unexecuted_code_cells"] > 0, "stage"].tolist())

**03. Verify the source-linked research evidence register**

The register contains one explicit research question, exact notebook filename, experimental scope, principal previously reported observation and interpretation restriction per stage. It deliberately does **not** fabricate a machine-extracted uniform score table: protocols differ and many notebooks contain diagnostics rather than comparable model scores. Inspect the full register and focus on the rationale and limitations for each stage.

In [ ]:
# Enforce one source-matched register record for each notebook stage.
register = pd.read_csv(register_path, encoding="utf-8")
evidence = validate_evidence_register(inventory, register)
assert evidence["stage"].tolist() == list(range(1, 20))
display(evidence[[
    "stage", "research_question", "protocol",
    "key_evidence", "limitations",
]])
print("Evidence register and 19 source notebooks: ALIGNED")

**04. Trace the development dataset and original subject boundaries**

**Source:** official HARTH v2.0 recordings from the pinned public source revision. Six accelerometer channels (back and thigh, three axes each) were sampled nominally at 50 Hz. The windowing protocol uses 100 samples (2 s), a 50-sample stride (50% overlap), annotation purity and timestamp continuity.

**Task restriction:** supervised models use eight documented activities (`1`–`8`). Cycling codes and undefined/unmapped source codes are counted and excluded *after* time-safe segmentation; their identities must not be guessed. The complete development cohort has 29 participants and 132,397 eligible windows for the eight-class task. Bounded deep-learning studies use 14,201 original 500-cap windows; nested 1,000-cap studies have 28,201 source-window rows. The original protected Test IDs are `S023` and `S032`, but their early descriptive inspection means no claim of completely unseen, preregistered Test is warranted.

In [ ]:
# Document verified counts from the executed notebooks, not new model estimates.
dataset_facts = pd.DataFrame([
    {"cohort": "Full development, 29 people",
     "eligible_windows": 132397, "channels": 6,
     "time_steps": 100, "source_stage": 8},
    {"cohort": "Original 500-cap development sample",
     "eligible_windows": 14201, "channels": 6,
     "time_steps": 100, "source_stage": 10},
    {"cohort": "Nested 1000-cap development sample",
     "eligible_windows": 28201, "channels": 6,
     "time_steps": 100, "source_stage": 15},
])
display(dataset_facts)
assert (dataset_facts["eligible_windows"] > 0).all()
print("Reserved original Test IDs: S023, S032")
print("Reserved from later model development: YES")
print("Never previously inspected: NO")

**05. Identify experiments that are directly comparable**

**Matched comparisons:** Stages 11–12 share the exact bounded 500-cap single Train/Validation split; Stage 13 compares three models under identical 3-fold held-out participant assignments; Stages 15–16 compare 500-cap versus strictly nested 1,000-cap Train while holding each paired Validation source row fixed; Stage 17 controls 400 optimiser updates across these training sizes.

**Not matched:** Stage 09 full-cohort classical CV uses 132,397 windows and five folds and cannot be compared as though it were the same benchmark as the 14,201-window CNN pilot in Stage 10. Stage 19 changes the training design by introducing a separate group of Calibration participants; its raw scores are not direct replications of Stage 18.

In [ ]:
# Provide the protocol boundary rather than an invalid cross-stage leaderboard.
comparison_map = pd.DataFrame([
    {"stages": "09 vs 10", "matched": False,
     "reason": "Full 132,397-window, 5-fold classical CV vs capped 14,201-window, one-split CNN"},
    {"stages": "11–12", "matched": True,
     "reason": "Identical original 500-cap source windows and six-person Validation split"},
    {"stages": "13", "matched": True,
     "reason": "Three classifiers evaluated on the same 3 GroupKFold assignments and 500-cap rows"},
    {"stages": "15", "matched": True,
     "reason": "Strictly nested Train windows; six-person Validation source rows fixed"},
    {"stages": "16", "matched": True,
     "reason": "Four training-size/model conditions use identical base Validation rows within folds"},
    {"stages": "17", "matched": True,
     "reason": "Two nested Train conditions, same 400 updates and same held-out people/rows"},
    {"stages": "18 vs 19", "matched": False,
     "reason": "Stage 19 withholds separate Calibration subjects from CNN Train"},
])
display(comparison_map)

**06. Consolidate the interpretable numerical comparisons**

The following tables are **source-reported summaries**, not newly trained or pooled predictions. Fixed eight-class Macro F1 is the main classification outcome. Read results within each matched group only. Avoid statistical superiority claims based on 3-fold means or 29 correlated groups; leave different training budgets and representations explicit.

In [ ]:
# Summaries copied from the saved Stage 09, 13, 16 and 17 output tables.
grouped_scores = pd.DataFrame([
    {"stage": 9, "protocol": "full 29-person, 5 folds, 132397 windows",
     "model": "Random Forest", "mean_macro_f1": 0.8339},
    {"stage": 9, "protocol": "full 29-person, 5 folds, 132397 windows",
     "model": "Logistic Regression", "mean_macro_f1": 0.7710},
    {"stage": 13, "protocol": "bounded 500-cap, 3 folds",
     "model": "Random Forest", "mean_macro_f1": 0.8076},
    {"stage": 13, "protocol": "bounded 500-cap, 3 folds",
     "model": "Standard CNN", "mean_macro_f1": 0.6301},
    {"stage": 13, "protocol": "bounded 500-cap, 3 folds",
     "model": "Weighted CNN", "mean_macro_f1": 0.7834},
    {"stage": 16, "protocol": "nested Train, matched 500-cap Validation, 3 folds, fixed epochs",
     "model": "RF 500", "mean_macro_f1": 0.8076},
    {"stage": 16, "protocol": "nested Train, matched 500-cap Validation, 3 folds, fixed epochs",
     "model": "RF 1000", "mean_macro_f1": 0.8097},
    {"stage": 16, "protocol": "nested Train, matched 500-cap Validation, 3 folds, fixed epochs",
     "model": "Weighted CNN 500", "mean_macro_f1": 0.7834},
    {"stage": 16, "protocol": "nested Train, matched 500-cap Validation, 3 folds, fixed epochs",
     "model": "Weighted CNN 1000", "mean_macro_f1": 0.7962},
    {"stage": 17, "protocol": "nested Train, matched 500-cap Validation, 3 folds, fixed 400 updates",
     "model": "Weighted CNN 500", "mean_macro_f1": 0.7746},
    {"stage": 17, "protocol": "nested Train, matched 500-cap Validation, 3 folds, fixed 400 updates",
     "model": "Weighted CNN 1000", "mean_macro_f1": 0.7927},
])
display(grouped_scores)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, stage in zip(axes, [13, 17]):
    group = grouped_scores[grouped_scores["stage"] == stage]
    ax.barh(group["model"], group["mean_macro_f1"])
    ax.set(title=f"Stage {stage} | matched development protocol",
           xlabel="Mean held-out-fold Macro F1", xlim=(0, 1))
    ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
plt.show()

**07. Review minority-class evidence and sampling sensitivity**

Shuffling has repeatedly been a difficult class, but that does not establish a single causal explanation. The source-audited full development cohort contains 1,946 Shuffling windows. The original cap-500 policy retains 219 and a separately even-spaced cap-1000 policy retains 386. The later *nested* cap-1000 policy is not the same selection as Stage 14's independent 1000-point linspace, so its counts must be reported separately.

Across Stage 13's matched three folds, pooled Shuffling F1 was 0.4957 for Random Forest and 0.1345 for Weighted CNN. Stage 16's Weighted CNN 1000 condition increased pooled Shuffling F1 to 0.2513 on the same base-cap held-out candidates; this is an exploratory observation, not proof that additional training data caused the effect.

In [ ]:
# Summarise verified retention; do not substitute data-derived model efficacy.
rare_support = pd.DataFrame([
    {"activity": "Shuffling", "full": 1946, "cap_500": 219, "cap_1000_independent": 386},
    {"activity": "Stairs ascending", "full": 1332, "cap_500": 192, "cap_1000_independent": 375},
    {"activity": "Stairs descending", "full": 1066, "cap_500": 182, "cap_1000_independent": 348},
])
rare_support["retained_500_pct"] = rare_support["cap_500"] / rare_support["full"] * 100
rare_support["retained_1000_pct"] = rare_support["cap_1000_independent"] / rare_support["full"] * 100
display(rare_support.round(2))
ax = rare_support.set_index("activity")[["retained_500_pct", "retained_1000_pct"]].plot(
    kind="bar", figsize=(11, 4.5), rot=0)
ax.set(title="Documented minority activities retained by chronological sampling",
       ylabel="Percentage of full eligible windows")
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

**08. Interpret softmax confidence and calibration separately**

Stage 18 audits an uncalibrated model on 14,201 out-of-fold development windows and finds ECE 0.0453, multiclass Brier 0.0905 and 40 wrong predictions with softmax ≥ 0.90. Stage 19 uses a different Train/Calibration/Validation design with non-overlapping participants. Mean ECE improves from 0.0497 to 0.0286 and Brier from 0.1088 to 0.1047 after temperature scaling, but mean NLL rises from 0.2502 to 0.2587. Two of three folds improve NLL and Brier and one worsens them. Temperature Scaling does not change the predicted top-1 class and must not be described as fixing Shuffling recall.

In [ ]:
# Keep the paired Stage 19 metrics separate from Stage 18.
calibration = pd.DataFrame([
    {"metric": "ECE (10 bins)", "uncalibrated": 0.0497, "temperature_scaled": 0.0286},
    {"metric": "Multiclass Brier", "uncalibrated": 0.1088, "temperature_scaled": 0.1047},
    {"metric": "NLL", "uncalibrated": 0.2502, "temperature_scaled": 0.2587},
    {"metric": "Accuracy", "uncalibrated": 0.9271, "temperature_scaled": 0.9271},
])
calibration["change_after_minus_before"] = (
    calibration["temperature_scaled"] - calibration["uncalibrated"]
)
display(calibration.round(4))
print("Calibration benefits are metric-dependent and fold-dependent.")

**09. Document risks, gaps and boundaries for future work**

**Label provenance:** Unmapped codes `10`, `11` and `17` appear in inspected source files. They have not been assigned activity names, and examples with these labels are excluded after windowing. Cycling and undefined codes must not be silently merged with the eight-class task.

**Sampling and validity:** Chronologically overlapping windows are correlated. Many participants are missing some task classes. Small out-of-fold class supports should be shown alongside F1. GroupKFold is subject-disjoint but does not guarantee class-stratified validation. Source data from the same study do not constitute external validation.

**Test boundary:** `S023` and `S032` must stay out of further model selection and training; their earlier descriptive use should remain disclosed.

**Performance claims:** Different caps, update budgets and calibration roles cannot be collapsed into a single model ranking. No clinical deployment, activity-monitoring validity in new settings, or final independent generalisation has been shown.

In [ ]:
# Present concrete corrective actions, prioritised before future architecture work.
research_gaps = pd.DataFrame([
    {"priority": "Essential", "topic": "Unmapped source codes",
     "next_action": "Verify source documentation; preserve raw IDs and exclusion audit."},
    {"priority": "Essential", "topic": "Minority-class class support",
     "next_action": "Assess Shuffling/stairs errors by participant, with counts and source windows."},
    {"priority": "Essential", "topic": "Cross-stage comparability",
     "next_action": "Use identical source rows, folds, class denominator and reported budgets."},
    {"priority": "Essential", "topic": "Reserved Test provenance",
     "next_action": "Freeze protocol first; disclose early exploratory inspection."},
    {"priority": "Essential", "topic": "Outdated README and roadmap",
     "next_action": "Synchronise documentation with executed experiment evidence."},
    {"priority": "Planned", "topic": "Advanced temporal architecture benchmark",
     "next_action": "Implement TCN, Multi-Scale CNN, 1D-ResNet and Transformer in Stage 22."},
])
display(research_gaps)

**10. Stage 20 reproducibility checks and hand-off to Stage 21**

This quality gate checks source filenames, inventory coverage, source-stage references and internal score ranges. It does **not** reproduce 19 expensive training runs. The evidence register should remain version-controlled, linked to the executed notebooks and re-reviewed if source notebooks change.

**Stage 21 hand-off:** Inspect per-subject class support, specific error types and ambiguous source codes in detail. Do not introduce new architectures before that investigation; Stages 22–23 will implement and compare all four agreed temporal model families.

In [ ]:
# Verify that every stage has saved code, every registered source exists
# and no reported summary has impossible Macro F1 values.
assert len(inventory) == 19
assert set(inventory["stage"]) == set(range(1, 20))
assert not inventory["saved_error_count"].gt(0).any()
assert all((grouped_scores["mean_macro_f1"] >= 0) &
           (grouped_scores["mean_macro_f1"] <= 1))
assert (rare_support[["cap_500", "cap_1000_independent"]]
        .le(rare_support["full"], axis=0)).all().all()
assert evidence["stage"].nunique() == 19
print("Stage 20 source inventory and research-register checks: PASSED")
print("Notebook reruns or model fitting performed: NO")
print("Independent Test evaluation established: NO")
print("Stage 21: minority-class and source-label quality review")